# Notebook 04: Physics RLVR — Merge Checkpoint 250

Merge the GRPO adapter at `outputs/checkpoint-250` into the base model in **float32**, then save a standalone **float32** model for reevaluation.


## Step 1: Mount Google Drive


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
%cd /content/drive/MyDrive/grpo-verified-reasoner
!ls


In [ ]:
!nvidia-smi


## Step 2: Install Dependencies and Import Libraries

Use the same `unsloth` / `unsloth-zoo` pin as the GRPO notebook so merge behavior matches the training environment.


In [ ]:
# Install UV (faster pip)
!pip install --upgrade -qqq uv


In [ ]:
import os
import torch
import subprocess

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    try:
        import numpy, PIL
        get_numpy = f"numpy=={numpy.__version__}"
        get_pil = f"pillow=={PIL.__version__}"
    except Exception:
        get_numpy = "numpy"
        get_pil = "pillow"

    !uv pip install -qqq --upgrade         unsloth {get_numpy} {get_pil} torchvision bitsandbytes xformers

# Match the exact stack used for GRPO training
!uv pip install -qqq transformers==4.56.2
!uv pip install -qqq --no-deps --force-reinstall unsloth==2026.1.2 unsloth-zoo==2026.1.2


In [ ]:
import os
import torch
from safetensors import safe_open
from unsloth import FastLanguageModel

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Step 3: Define Paths and Verify Checkpoint


In [ ]:
CHECKPOINT_PATH = "outputs/checkpoint-250"
MODEL_OUT = "physics_rlvr/models/qwen3-4b-physics-grpo-checkpoint-250-merged-32bit"
MAX_SEQ_LENGTH = 3072

assert os.path.isdir(CHECKPOINT_PATH), f"Checkpoint not found: {CHECKPOINT_PATH}"
os.makedirs(MODEL_OUT, exist_ok=True)

print(f"Checkpoint found: {CHECKPOINT_PATH}")
print("Checkpoint files:", sorted(os.listdir(CHECKPOINT_PATH)))
print(f"Output folder: {MODEL_OUT}")


## Step 4: Load Checkpoint in Float32 and Merge

Load the checkpoint adapter in float32 so the merge is performed in full precision.


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = CHECKPOINT_PATH,
    max_seq_length  = MAX_SEQ_LENGTH,
    load_in_4bit    = False,
    dtype           = torch.float32,
)


In [ ]:
# Apply the LoRA adapter deltas into the base weights in-place.
# Because the model is loaded in float32, the merge preserves small RL updates.
model.merge_and_unload()


## Step 5: Save the Standalone Float32 Model

Use Hugging Face's native `save_pretrained` on the underlying model to preserve the full float32 weights and write a standard standalone model directory (including `config.json`).


In [ ]:
model.model.save_pretrained(MODEL_OUT, safe_serialization=True)
tokenizer.save_pretrained(MODEL_OUT)

print(f"Merged model saved to: {MODEL_OUT}")
print("Files saved:", sorted(os.listdir(MODEL_OUT)))


## Step 6: Verify Saved Precision

Inspect the first tensor in the saved safetensors shard to confirm the exported model is float32.


In [ ]:
weight_files = sorted([
    f for f in os.listdir(MODEL_OUT)
    if f.endswith('.safetensors')
])
assert len(weight_files) > 0, "No safetensors weight files found."

first_weight = os.path.join(MODEL_OUT, weight_files[0])
with safe_open(first_weight, framework="pt", device="cpu") as f:
    first_key = next(iter(f.keys()))
    first_dtype = f.get_tensor(first_key).dtype

print("First shard:", first_weight)
print("First tensor key:", first_key)
print("First tensor dtype:", first_dtype)
assert first_dtype == torch.float32, f"Expected float32, got {first_dtype}"
print("Verification passed: merged model is saved in float32.")


## Step 7: Next Step

Point the evaluation notebook to:

`physics_rlvr/models/qwen3-4b-physics-grpo-checkpoint-250-merged-32bit`

and rerun only the GRPO evaluation cell plus the summary cells.
